# W5D4 — Attention and the Transformer — Lab

**Week 5 · Day 4 · NLP Foundations** · Lab

Two kinds of memory so far. The RNN carried a state from token to token, and its memory of the first
token faded with every step. The TCN looked through a fixed window and saw nothing outside it.

Attention removes both limits. Every token scores every other token, turns the scores into weights
with a softmax, and takes a weighted sum — **one step, any distance**. You compute it once by hand, on
the three vectors the RNN read on Tuesday. After that you use the version PyTorch ships, twice:

- an **encoder**, where every token sees the whole review, to classify sentiment;
- a **decoder**, where every token sees only the past, to *write* reviews one character at a time.

Same block both times. The only difference is one mask.

**Time budget:** ~115 minutes. Sections 1–3 are the lab; Section 4 is a stretch you may finish at home.

<div dir="rtl" align="right">

# الأسبوع ٥ · اليوم ٤ — الانتباه والمحوّل

**الأسبوع الخامس · اليوم الرابع · أساسيات معالجة اللغة** · معمل

نوعان من الذاكرة حتى الآن. حملت RNN حالةً من رمز إلى رمز، فبهتت ذاكرتها للرمز الأول مع كل خطوة. ونظرت TCN عبر
نافذة ثابتة فلم ترَ شيئًا خارجها.

والانتباه يرفع القيدين كليهما. كل رمز يُقيّم كل رمز آخر، ويحوّل الدرجات إلى أوزان بـsoftmax، ويأخذ مجموعًا موزونًا
— **خطوة واحدة، وأي مسافة**. تحسبه مرّةً يدويًا، على المتّجهات الثلاثة التي قرأتها RNN يوم الثلاثاء. وبعدها تستعمل
النسخة التي يوفّرها PyTorch، مرّتين:

- **مُرمِّزًا** (encoder)، يرى فيه كل رمز المراجعة كلها، لتصنيف المشاعر؛
- **مُفكِّكًا** (decoder)، لا يرى فيه كل رمز إلا الماضي، لـ*كتابة* المراجعات حرفًا بعد حرف.

الكتلة نفسها في المرّتين. والفرق الوحيد قناعٌ واحد.

**الزمن المتوقّع:** نحو ١١٥ دقيقة. الأقسام ١–٣ هي المعمل، والقسم الرابع إضافي يمكن إكماله في المنزل.

</div>

> **This is your lab notebook.** Work through the hints — they tell you what to do and where
> to look, not what to type. Stuck for more than ten minutes on one task? Open the `_guided`
> version. That is not cheating; sitting stuck in silence is the only mistake. The full
> solution is released at the end of the day.

<div dir="rtl" align="right">

> **هذا دفتر المعمل الخاص بك.** اعمل وفق الإرشادات — فهي تخبرك بما يجب فعله وأين تبحث، لا بما
> تكتبه حرفيًا. إذا توقّفت أكثر من عشر دقائق عند مهمة واحدة فافتح نسخة `_guided`؛ هذا ليس غشًّا،
> والخطأ الوحيد هو أن تبقى متوقّفًا بصمت. ويُنشر الحل الكامل في نهاية اليوم.

</div>

## Learning objectives

By the end of this lab you can:

- Compute attention by hand — scores, softmax, weighted sum — and reproduce the slide's numbers.
- Explain why attention alone cannot see order, and add positions to fix it.
- Use `nn.MultiheadAttention` and `nn.TransformerEncoder`, and read what goes in and what comes out.
- Train a transformer **encoder** to classify reviews, with the padding masked out.
- Explain the **causal mask**, and train a **decoder-only** language model that writes new text.

<div dir="rtl" align="right">

## أهداف التعلّم

في نهاية هذا المعمل تستطيع:

- أن تحسب الانتباه يدويًا — الدرجات وsoftmax والمجموع الموزون — وتُعيد إنتاج أرقام الشريحة.
- أن تشرح لماذا لا يرى الانتباه وحده الترتيب، وتضيف المواضع لإصلاح ذلك.
- أن تستعمل `nn.MultiheadAttention` و`nn.TransformerEncoder`، وتقرأ ما يدخل وما يخرج.
- أن تُدرّب **مُرمِّز** محوّل على تصنيف المراجعات، مع إخفاء الحشو.
- أن تشرح **القناع السببي**، وتُدرّب نموذج لغة **مُفكِّكًا فقط** (decoder-only) يكتب نصًا جديدًا.

</div>

## About the data

The warm-up uses **three hard-coded vectors**, on purpose: no data loading between you and the
mechanism.

Both halves then use `reviews_sentiment` on the week's split (`week5_split.json`, hash checked):

- **the encoder** reads reviews tokenised exactly as on Tuesday (20,000 words, padded to 128) and
  adds today's row to the leaderboard;
- **the decoder** reads the English training reviews as one long string of characters — 44 different
  characters in all — and learns to continue it. The test reviews are kept back to measure it.

Nothing downloads today. Two cells train for about a minute each: the review encoder and the language
model.

<div dir="rtl" align="right">

## عن البيانات

يستعمل الإحماء **ثلاثة متّجهات مكتوبة في الشيفرة**، عمدًا: لا تحميل بيانات بينك وبين الآلية.

ثم يستعمل النصفان `reviews_sentiment` على تقسيم الأسبوع (`week5_split.json` مع التحقّق من البصمة):

- **المُرمِّز** يقرأ المراجعات مُرمَّزةً تمامًا كيوم الثلاثاء (٢٠٬٠٠٠ كلمة، والحشو حتى ١٢٨) ويضيف صفّ اليوم إلى اللوحة؛
- **المُفكِّك** يقرأ مراجعات التدريب الإنجليزية سلسلةً واحدة طويلة من الحروف — ٤٤ حرفًا مختلفًا في المجموع — ويتعلّم
  أن يُكملها. وتُحفظ مراجعات الاختبار جانبًا لقياسه.

لا يُنزَّل شيء اليوم. وخليّتان تتدرّبان نحو دقيقة لكلٍّ منهما: مُرمِّز المراجعات ونموذج اللغة.

</div>

## Setup

<div dir="rtl" align="right">

## الإعداد

</div>

In [ ]:
# === AIEP portable setup — works locally (conda) and on Google Colab ===============
try:
    import aiep
except ImportError:
    import subprocess, sys
    from pathlib import Path
    _local = next((p / "shared" for p in [Path.cwd(), *Path.cwd().parents]
                   if (p / "shared" / "aiep").is_dir()), None)
    if _local:
        sys.path.insert(0, str(_local))
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "git+https://github.com/0xRush/AIEP_Olo_student.git#subdirectory=shared"])
    import aiep

from aiep.env import ensure, seed_everything, versions
from aiep.data import get_dataset, load_artefact
from aiep.paths import ARTEFACT_DIR
from aiep.checks import check, check_close, report

ensure("matplotlib")
seed_everything(42)

import collections
import hashlib
import json
import math
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

SEED = 42
torch.set_num_threads(4)
np.set_printoptions(precision=4, suppress=True)

# The three tokens from this morning's slide — and from Tuesday's RNN — and the query.
V = np.array([[1.0, 0.0],
              [0.0, 1.0],
              [1.0, 1.0]])
QUERY = np.array([1.0, 0.0])
print(versions())

## Section 1 — Warm-up: attention, by hand  (≈20 min)

Everything here works.

The slide's example, step by step:

1. Three scores, one dot product each: `q · v₁ = 1`, `q · v₂ = 0`, `q · v₃ = 1`.
2. Softmax: `e¹, e⁰, e¹` = 2.72, 1.00, 2.72, sum 6.44 → weights **0.42, 0.16, 0.42**.
3. Weighted sum: `0.42·[1,0] + 0.16·[0,1] + 0.42·[1,1]` = **`[0.84, 0.58]`**.

The softmax subtracts the largest score first. That changes nothing mathematically and keeps
`exp` from overflowing on large scores — try `[1000, 1001]` without it.

Then change the query to `[0, 1]`: the weights flip to **0.16, 0.42, 0.42**. The same tokens, read
differently, because the question changed.

<div dir="rtl" align="right">

## القسم الأول — الإحماء: الانتباه، يدويًا (نحو ٢٠ دقيقة)

كل ما هنا يعمل.

مثال الشريحة خطوةً خطوة:

١. ثلاث درجات، جداء نقطي لكلٍّ منها: `q · v₁ = 1` و`q · v₂ = 0` و`q · v₃ = 1`.
٢. softmax: `e¹, e⁰, e¹` = ٢٫٧٢ و١٫٠٠ و٢٫٧٢، ومجموعها ٦٫٤٤ ← الأوزان **٠٫٤٢ و٠٫١٦ و٠٫٤٢**.
٣. المجموع الموزون: `0.42·[1,0] + 0.16·[0,1] + 0.42·[1,1]` = **`[0.84, 0.58]`**.

تطرح softmax أكبر درجة أولًا. وهذا لا يغيّر شيئًا رياضيًا ويمنع `exp` من الطفح على الدرجات الكبيرة — جرّب
`[1000, 1001]` بدونه.

ثم غيّر المُستعلِم إلى `[0, 1]`: فتنقلب الأوزان إلى **٠٫١٦ و٠٫٤٢ و٠٫٤٢**. الرموز نفسها تُقرأ بطريقة مختلفة،
لأن السؤال تغيّر.

</div>

In [ ]:
def softmax(scores, axis=-1):
    """Numerically stable softmax: subtract the max, exponentiate, normalise."""
    shifted = scores - np.max(scores, axis=axis, keepdims=True)
    exponentials = np.exp(shifted)
    return exponentials / exponentials.sum(axis=axis, keepdims=True)


def attention(q, values):
    """Single-query attention. Returns (output, weights)."""
    weights = softmax(values @ q)
    return weights @ values, weights


OUTPUT, WEIGHTS = attention(QUERY, V)
FLIPPED_OUTPUT, FLIPPED_WEIGHTS = attention(np.array([0.0, 1.0]), V)
print(f"scores  : {V @ QUERY}")
print(f"weights : {WEIGHTS.round(2)}        output {OUTPUT.round(2)}")
print(f"flipped : {FLIPPED_WEIGHTS.round(2)}        output {FLIPPED_OUTPUT.round(2)}")
with np.errstate(over="ignore", invalid="ignore"):
    print(f"\n[1000, 1001] naive : {np.exp([1000.0, 1001.0]) / np.exp([1000.0, 1001.0]).sum()}")
print(f"[1000, 1001] stable: {softmax(np.array([1000.0, 1001.0])).round(4)}")

### Attention is blind to order

Feed the same three tokens in **reverse** order with the same query. On Tuesday the RNN's final state
changed when you did this. Here the output is **bit-for-bit identical**: each weight depends only on
its own token, and a sum does not care about the order of its terms. Position is nowhere in the
arithmetic — so every transformer adds it to the input. The original paper used fixed sine waves; the
models you will build today simply **learn** one vector per position, with an `nn.Embedding`, as GPT
does.

<div dir="rtl" align="right">

### الانتباه أعمى عن الترتيب

أدخِل الرموز الثلاثة نفسها بترتيب **معكوس** بالمُستعلِم نفسه. تغيّرت الحالة الأخيرة لـRNN يوم الثلاثاء حين
فعلت هذا. أمّا هنا فالمخرَج **متطابق بتًّا ببت**: فكل وزن يعتمد على رمزه وحده، والمجموع لا يبالي بترتيب حدوده.
والموضع ليس في الحساب في أي مكان — ولذلك يضيفه كل محوّل إلى المُدخل. استعملت الورقة الأصلية موجات جيبية ثابتة؛
أمّا النماذج التي تبنيها اليوم فـ**تتعلّم** متّجهًا لكل موضع، بـ`nn.Embedding`، كما يفعل GPT.

</div>

In [ ]:
REVERSED_OUTPUT, REVERSED_WEIGHTS = attention(QUERY, V[::-1])
ORDER_BLIND = np.array_equal(OUTPUT, REVERSED_OUTPUT)
print(f"in order : weights {WEIGHTS.round(4)} -> output {OUTPUT}")
print(f"reversed : weights {REVERSED_WEIGHTS.round(4)} -> output {REVERSED_OUTPUT}")
print(f"identical output: {ORDER_BLIND}")

## Section 2 — The encoder: reading a review  (≈35 min)

An encoder lets every token look at every other token, before and after it. That is what you want
when the whole text is already there and you need one answer about it — like its sentiment.

1. `nn.MultiheadAttention` gives back the slide's numbers.
2. The encoder layer: what is inside, and what comes out.
3. A transformer encoder on the reviews.

<div dir="rtl" align="right">

## القسم الثاني — المُرمِّز: قراءة مراجعة (نحو ٣٥ دقيقة)

يتيح المُرمِّز لكل رمز أن ينظر إلى كل رمز آخر، قبله وبعده. وهذا ما تريده حين يكون النص كله موجودًا وتحتاج جوابًا
واحدًا عنه — كمشاعره.

١. `nn.MultiheadAttention` تُعيد أرقام الشريحة.
٢. طبقة المُرمِّز: ما بداخلها، وما يخرج منها.
٣. مُرمِّز محوّل على المراجعات.

</div>

### Task 2.1 — `nn.MultiheadAttention` gives back the slide

Inside `nn.MultiheadAttention` is the formula you just used, for many queries at once:

`attention(Q, K, V) = softmax(Q Kᵀ / √d) V`

Around it, the module adds learned projections for the queries, keys and values. The cell below
switches them off — one head, every projection set to "change nothing" — so only the formula is left.
Call it with the warm-up's two queries, and the slide's vectors as keys and values.

Row 0 is the warm-up again, a little softer: **0.40, 0.20, 0.40**. The softening is the `/ √d` (here
`√2`). In a big model `d` is large, the scores get large, and without it the softmax would put almost
all the weight on one token.

<div dir="rtl" align="right">

### المهمة ٢٫١ — `nn.MultiheadAttention` تُعيد الشريحة

داخل `nn.MultiheadAttention` الصيغة التي استعملتها للتوّ، لمُستعلِمات كثيرة دفعةً واحدة:

`attention(Q, K, V) = softmax(Q Kᵀ / √d) V`

وحولها تضيف الوحدة إسقاطات مُتعلَّمة للمُستعلِمات والمفاتيح والقيم. والخلية أدناه تُطفئها — رأس واحد، وكل إسقاط
مضبوط على «لا تغيّر شيئًا» — فلا يبقى إلا الصيغة. استدعِها بمُستعلِمَي الإحماء، ومتّجهات الشريحة مفاتيحَ وقيمًا.

فالصف ٠ هو الإحماء مرّة أخرى، ألين قليلًا: **٠٫٤٠ و٠٫٢٠ و٠٫٤٠**. والتليين هو `/ √d` (هنا `√2`). ففي نموذج كبير
يكون `d` كبيرًا، فتكبر الدرجات، ولولا القسمة لوضعت softmax الوزن كله تقريبًا على رمز واحد.

</div>

In [ ]:
slide = torch.tensor(V, dtype=torch.float32)[None]           # (batch 1, 3 tokens, 2 numbers each)
queries = torch.tensor([[[1.0, 0.0], [0.0, 1.0]]])              # (batch 1, 2 queries, 2): the warm-up's and the flipped one

plain_attention = nn.MultiheadAttention(embed_dim=2, num_heads=1, batch_first=True)
with torch.no_grad():                                          # switch the learned projections off
    plain_attention.in_proj_weight.copy_(torch.eye(2).repeat(3, 1))
    plain_attention.in_proj_bias.zero_()
    plain_attention.out_proj.weight.copy_(torch.eye(2))
    plain_attention.out_proj.bias.zero_()

# TODO: Call plain_attention with (queries, slide, slide). It returns (output, weights).
# مهمة: استدعِ `plain_attention` بـ`(queries, slide, slide)`. وهي تُعيد `(output, weights)`.

print("weights, one row per query:")
print(SLIDE_WEIGHTS[0].detach().numpy().round(2))
SLIDE_MATCH = np.allclose(SLIDE_WEIGHTS[0, 0].detach().numpy(), softmax(V @ QUERY / np.sqrt(2)), atol=1e-6)
print(f"row 0 = the warm-up's softmax of [1, 0, 1] / sqrt(2): {SLIDE_MATCH}")

### Task 2.2 — the encoder layer

A transformer layer is attention plus three pieces you already know:

1. **attention**, then a **residual** connection (`x + attention(x)`) — ResNet's skip connection;
2. **LayerNorm**;
3. a small **feed-forward** network applied to each token on its own (`Linear → GELU → Linear`);
4. a second residual and a second LayerNorm.

PyTorch packages all of it as `nn.TransformerEncoderLayer`, and stacks layers with
`nn.TransformerEncoder`. Print one layer and find the pieces: `self_attn`, `norm1`, `linear1` and
`linear2`, `norm2`. Then push a batch through two stacked layers. What goes in is
`(batch, tokens, d_model)` — and what comes out has **exactly the same shape**: one updated vector per
token, each now mixed with information from the others. That is why layers stack.

Two words you will set every time: `d_model`, the length of each token's vector, and `nhead`, how many
attentions run side by side. `d_model` must divide by `nhead`.

<div dir="rtl" align="right">

### المهمة ٢٫٢ — طبقة المُرمِّز

طبقة المحوّل هي الانتباه ومعه ثلاث قطع تعرفها:

١. **الانتباه**، ثم وصلة **متبقّية** (`x + attention(x)`) — وصلة التخطّي في ResNet؛
٢. **LayerNorm**؛
٣. شبكة **تغذية أمامية** صغيرة تُطبَّق على كل رمز وحده (`Linear ← GELU ← Linear`)؛
٤. متبقّية ثانية وLayerNorm ثانية.

ويجمع PyTorch ذلك كله في `nn.TransformerEncoderLayer`، ويُكدّس الطبقات بـ`nn.TransformerEncoder`. اطبع طبقة واحدة
وجِد القطع: `self_attn`، و`norm1`، و`linear1` و`linear2`، و`norm2`. ثم مرّر دفعةً عبر طبقتين مكدّستين. ما يدخل
`(batch, tokens, d_model)` — وما يخرج له **الشكل نفسه تمامًا**: متّجه مُحدَّث لكل رمز، وقد امتزج كلٌّ منها بمعلومات
من البقية. ولهذا تتكدّس الطبقات.

وكلمتان ستضبطهما كل مرّة: `d_model`، طول متّجه كل رمز، و`nhead`، عدد الانتباهات التي تعمل جنبًا إلى جنب. ويجب أن
يقبل `d_model` القسمة على `nhead`.

</div>

In [ ]:
layer = nn.TransformerEncoderLayer(d_model=64, nhead=4, dim_feedforward=256, dropout=0.0,
                                   activation="gelu", batch_first=True)
print(layer)

encoder = nn.TransformerEncoder(layer, num_layers=2, enable_nested_tensor=False)   # two copies, stacked
tokens_in = torch.randn(8, 20, 64)                              # 8 texts, 20 tokens each, 64 numbers per token

# TODO: Run tokens_in through the encoder and store the result in tokens_out.
# مهمة: مرّر `tokens_in` عبر `encoder` واحفظ النتيجة في `tokens_out`.
print(f"\nin  {tuple(tokens_in.shape)}\nout {tuple(tokens_out.shape)}")
print(f"{sum(p.numel() for p in encoder.parameters()):,} parameters in two layers")

### Task 2.3 — a transformer encoder on the reviews

Now the real thing. The classifier reads a review in four steps:

1. **embed** every word, and **add** the vector for its position;
2. run the **encoder** — two layers, `d_model = 64`, 4 heads;
3. **average** the word vectors into one vector for the review;
4. a `Linear` layer turns that into two scores: negative, positive.

The reviews are padded to 128 words, and padding must not take part. You tell the encoder where it is
with `src_key_padding_mask` — `True` where a review has already ended — so no word attends to it; and
you leave the padding out of the average. Write `forward`; the training loop is given. Three epochs take
about a minute. Then the leaderboard row and the two sentence pairs.

Expect it **not** to beat Tuesday's GRU — and perhaps to lose. A transformer trained from scratch on
9,600 reviews has no advantage; its strengths are reaching any distance in one step, and training in
parallel on enormous data. Tomorrow's model has exactly that.

<div dir="rtl" align="right">

### المهمة ٢٫٣ — مُرمِّز محوّل على المراجعات

والآن الشيء الحقيقي. يقرأ المُصنّف المراجعة في أربع خطوات:

١. **ضمِّن** كل كلمة، و**أضِف** متّجه موضعها؛
٢. شغّل **المُرمِّز** — طبقتان، و`d_model = 64`، و٤ رؤوس؛
٣. **احسب متوسط** متّجهات الكلمات في متّجه واحد للمراجعة؛
٤. طبقة `Linear` تحوّله إلى درجتين: سلبي، وإيجابي.

والمراجعات محشوّة حتى ١٢٨ كلمة، ويجب ألّا يشارك الحشو. فتُخبر المُرمِّز بمكانه عبر `src_key_padding_mask` — `True`
حيث انتهت المراجعة — فلا تنتبه إليه أي كلمة؛ وتُبقي الحشو خارج المتوسط. اكتب `forward`؛ وحلقة التدريب معطاة. وتستغرق
ثلاث حقب نحو دقيقة. ثم صفّ اللوحة وزوجا الجمل.

وتوقّع **ألّا** يتفوّق على GRU الثلاثاء — وقد يخسر. فالمحوّل المُدرَّب من الصفر على ٩٬٦٠٠ مراجعة لا ميزة له؛ وقوّته
في بلوغ أي مسافة بخطوة واحدة، وفي التدريب على التوازي على بيانات هائلة. ونموذج الغد عنده ذلك بالضبط.

</div>

In [ ]:
MAX_LEN, VOCAB_SIZE = 128, 20_000
NEGATION_PAIR = ["the service was good", "the service was not good"]
ORDER_PAIR = ["the food was good but the service was bad",
              "the service was good but the food was bad"]
reviews = pd.read_parquet(get_dataset("reviews_sentiment"))
split = json.loads(load_artefact("week5_split.json").read_text(encoding="utf-8"))
check(hashlib.sha256(json.dumps({"train": split["train"], "test": split["test"]}).encode()).hexdigest()
      == split["sha256"], "week5_split.json must match its own hash", "يجب أن يطابق `week5_split.json` بصمته")
train, test = reviews.iloc[split["train"]], reviews.iloc[split["test"]]
print(f"train {len(train):,} | test {len(test):,} | split {split['sha256'][:12]}…")

def tokenize(text):
    return re.sub(r"[^0-9a-z؀-ۿ\s']+", " ", text.lower()).split()


counts = collections.Counter(w for text in train.review_text for w in tokenize(text))
ITOS = ["<pad>", "<unk>"] + [w for w, _ in counts.most_common(VOCAB_SIZE - 2)]
STOI = {w: i for i, w in enumerate(ITOS)}


def encode(texts):
    """Token ids, right-padded with 0 to MAX_LEN, and the true length of each review."""
    ids = [[STOI.get(w, 1) for w in tokenize(t)][:MAX_LEN] or [1] for t in texts]
    x = torch.zeros(len(ids), MAX_LEN, dtype=torch.long)
    for i, seq in enumerate(ids):
        x[i, :len(seq)] = torch.tensor(seq)
    return x, torch.tensor([len(seq) for seq in ids])


X_train, len_train = encode(train.review_text)
X_test, len_test = encode(test.review_text)
y_train = torch.tensor(train.sentiment.to_numpy()).long()
y_test = torch.tensor(test.sentiment.to_numpy()).long()
print(f"vocabulary {len(ITOS):,} | X_train {tuple(X_train.shape)} | longest review {int(len_train.max())} tokens")

def load_board(days):
    """The leaderboard rows earlier days wrote (each day keeps its own file)."""
    rows = []
    for day in days:
        rows += json.loads(load_artefact(f"leaderboard_{day}.json").read_text(encoding="utf-8"))
    return rows


def show_board(rows):
    columns = ["day", "model", "n_train", "accuracy", "f1", "fit_s", "ms_per_1000", "size_mb",
               "negation_ok", "order_gap"]
    print(pd.DataFrame(rows)[columns].to_string(index=False))

In [ ]:
class ReviewEncoder(nn.Module):
    def __init__(self, d_model=64, n_heads=4, n_layers=2):
        super().__init__()
        self.words = nn.Embedding(len(ITOS), d_model, padding_idx=0)
        self.positions = nn.Embedding(MAX_LEN, d_model)            # one learned vector per position
        layer = nn.TransformerEncoderLayer(d_model, n_heads, 4 * d_model, dropout=0.0,
                                           activation="gelu", batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.out = nn.Linear(d_model, 2)

    def forward(self, x, lengths):
        padding = torch.arange(x.shape[1]) >= lengths[:, None]      # (batch, words): True where the review has ended
        # TODO: Add word and position embeddings, run the encoder with src_key_padding_mask=padding, zero the padded positions, average over the real words, and classify.
        # مهمة: اجمع تضمين الكلمات وتضمين المواضع، وشغّل المُرمِّز مع `src_key_padding_mask=padding`، وصفّر المواضع المحشوّة، وخُذ المتوسط على الكلمات الحقيقية، وصنّف.


def accuracy(model):
    model.eval()
    with torch.no_grad():
        return float((model(X_test, len_test).argmax(1) == y_test).float().mean())


torch.manual_seed(SEED)
review_encoder = ReviewEncoder()
optimiser = torch.optim.Adam(review_encoder.parameters(), lr=1e-3)
start = time.perf_counter()
for epoch in range(3):
    review_encoder.train()
    order = torch.randperm(len(X_train))
    for i in range(0, len(order), 64):
        batch = order[i:i + 64]
        loss = nn.functional.cross_entropy(review_encoder(X_train[batch], len_train[batch]), y_train[batch])
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
    print(f"  epoch {epoch + 1}: test accuracy {accuracy(review_encoder):.4f}")
ENCODER_FIT_S = time.perf_counter() - start

In [ ]:
from sklearn.metrics import f1_score

review_encoder.eval()
start = time.perf_counter()
with torch.no_grad():
    encoder_predictions = review_encoder(X_test, len_test).argmax(1).numpy()
    ENCODER_MS_PER_1000 = (time.perf_counter() - start) / len(X_test) * 1e6
    x_pairs, l_pairs = encode(NEGATION_PAIR + ORDER_PAIR)
    p_pairs = torch.softmax(review_encoder(x_pairs, l_pairs), 1)[:, 1].numpy()

ENCODER_ACCURACY = accuracy(review_encoder)
BOARD_D4 = [{
    "day": "D4", "model": "transformer encoder (trained from scratch)", "n_train": len(X_train),
    "accuracy": round(ENCODER_ACCURACY, 4), "f1": round(f1_score(y_test.numpy(), encoder_predictions), 4),
    "fit_s": round(ENCODER_FIT_S, 1), "ms_per_1000": round(ENCODER_MS_PER_1000, 1),
    "size_mb": round(sum(p.numel() for p in review_encoder.parameters()) * 4 / 1e6, 2),
    "negation_ok": bool(p_pairs[0] > 0.5 > p_pairs[1]), "order_gap": round(float(abs(p_pairs[2] - p_pairs[3])), 4),
}]
for sentence, p in zip(NEGATION_PAIR + ORDER_PAIR, p_pairs):
    print(f"  p(positive) = {p:.4f}   {sentence}")
print()
show_board(load_board(["D1", "D2", "D3"]) + BOARD_D4)

## Section 3 — The decoder: writing a review  (≈35 min)

A **language model** does one thing: given the text so far, guess what comes next. Train it on
reviews, then feed its guesses back in, and it writes reviews. Tomorrow's LLM is exactly this, scaled up
a millionfold.

The model is the **same** encoder stack, with one change. When it guesses the character at position
`t + 1`, it must not see it — so every position may only look at itself and the positions before it.
That rule is the **causal mask**, and a stack with it is a **decoder-only** transformer, the design of
GPT.

1. The causal mask.
2. The data: every position's target is the next character.
3. Train a tiny GPT on the reviews.
4. Let it write.

<div dir="rtl" align="right">

## القسم الثالث — المُفكِّك: كتابة مراجعة (نحو ٣٥ دقيقة)

يفعل **نموذج اللغة** شيئًا واحدًا: بالنظر إلى النص حتى الآن، خمِّن ما يأتي بعده. درّبه على المراجعات، ثم أعِد إدخال
تخميناته إليه، فيكتب مراجعات. ونموذج اللغة الكبير غدًا هو هذا بالضبط، مكبَّرًا مليون مرّة.

والنموذج هو كوم المُرمِّز **نفسه**، بتغيير واحد. حين يُخمّن الحرف عند الموضع `t + 1` يجب ألّا يراه — فلا يجوز لأي
موضع أن ينظر إلا إلى نفسه وإلى المواضع قبله. وهذه القاعدة هي **القناع السببي**، والكوم الذي يحملها محوّل **مُفكِّك
فقط** (decoder-only)، وهو تصميم GPT.

١. القناع السببي.
٢. البيانات: هدف كل موضع هو الحرف التالي.
٣. درّب GPT صغيرًا على المراجعات.
٤. دعه يكتب.

</div>

### Task 3.1 — the causal mask

`nn.Transformer.generate_square_subsequent_mask(5)` builds the mask for five positions: `0` on and below
the diagonal, `−inf` above it. It is **added** to the attention scores before the softmax, and
`e^(−inf) = 0`, so every future position gets a weight of exactly zero.

Ask the encoder layer from task 2.2 for its attention weights with and without the mask. Without it,
every row spreads over all five positions. With it, row `t` only covers positions `0 … t` — the weights
form a triangle — and every row still sums to 1. It is Wednesday's causal padding again, in a new form.

<div dir="rtl" align="right">

### المهمة ٣٫١ — القناع السببي

يبني `nn.Transformer.generate_square_subsequent_mask(5)` القناع لخمسة مواضع: `0` على القُطر وتحته، و`−inf` فوقه.
ويُ**ضاف** إلى درجات الانتباه قبل softmax، و`e^(−inf) = 0`، فيأخذ كل موضع مستقبلي وزنًا صفرًا تمامًا.

اسأل طبقة المُرمِّز من المهمة ٢٫٢ عن أوزان انتباهها بالقناع وبدونه. بدونه يتوزّع كل صف على المواضع الخمسة كلها.
وبه لا يغطّي الصف `t` إلا المواضع `0 … t` — فتشكّل الأوزان مثلّثًا — ويبقى كل صف يجمع إلى ١. إنه الحشو السببي يوم
الأربعاء مرّة أخرى، بشكل جديد.

</div>

In [ ]:
CAUSAL_MASK = nn.Transformer.generate_square_subsequent_mask(5)
print(CAUSAL_MASK)

five_tokens = torch.randn(1, 5, 64)
_, open_weights = layer.self_attn(five_tokens, five_tokens, five_tokens)
# TODO: Ask layer.self_attn again with attn_mask=CAUSAL_MASK and keep the weights in causal_weights.
# مهمة: اسأل `layer.self_attn` مرّةً أخرى مع `attn_mask=CAUSAL_MASK` واحفظ الأوزان في `causal_weights`.

print("\nwithout the mask:\n", open_weights[0].detach().numpy().round(2))
print("\nwith the mask:\n", causal_weights[0].detach().numpy().round(2))
ABOVE_DIAGONAL = torch.triu(torch.ones(5, 5, dtype=torch.bool), diagonal=1)
IS_LOWER_TRIANGULAR = bool((causal_weights[0][ABOVE_DIAGONAL] == 0).all())
ROWS_SUM_TO_ONE = bool(torch.allclose(causal_weights.sum(-1), torch.ones(1), atol=1e-6))

### Task 3.2 — the data: guess the next character

The decoder works on **characters**, so there is no vocabulary to build: the English training reviews,
lower-cased and joined with a newline after each, make one long string of 44 different characters. Each
character becomes its number in that list.

A training example is a window of `CONTEXT = 64` characters, `x`, and its target `y` is **the same window
moved one character to the right**. So at every position the target is simply the next character, and
one window gives 64 guesses to learn from at once:

```
x:  t h e _ f o o d
y:  h e _ f o o d _
```

Write the line that cuts `y`.

<div dir="rtl" align="right">

### المهمة ٣٫٢ — البيانات: خمِّن الحرف التالي

يعمل المُفكِّك على **الحروف**، فلا معجم يُبنى: مراجعات التدريب الإنجليزية، بأحرف صغيرة، وبعد كلٍّ منها سطر جديد،
تصنع سلسلة واحدة طويلة من ٤٤ حرفًا مختلفًا. ويصير كل حرف رقمه في تلك القائمة.

ومثال التدريب نافذة من `CONTEXT = 64` حرفًا، `x`، وهدفها `y` هو **النافذة نفسها مُزاحةً حرفًا واحدًا إلى اليمين**.
فالهدف عند كل موضع هو الحرف التالي ببساطة، والنافذة الواحدة تعطي ٦٤ تخمينًا يُتعلَّم منها دفعةً واحدة:

```
x:  t h e _ f o o d
y:  h e _ f o o d _
```

اكتب السطر الذي يقصّ `y`.

</div>

In [ ]:
def as_text(texts):
    """English reviews only (the Arabic ones would add dozens of rare characters), one per line."""
    kept = [t.lower() for t in texts if re.fullmatch(r"[a-z0-9 .,!?'\-]+", t.lower())]
    return "\n".join(kept) + "\n"


train_text, test_text = as_text(train.review_text), as_text(test.review_text)
CHARS = sorted(set(train_text))
CHAR_ID = {c: i for i, c in enumerate(CHARS)}
train_ids = torch.tensor([CHAR_ID[c] for c in train_text])
test_ids = torch.tensor([CHAR_ID[c] for c in test_text if c in CHAR_ID])
CONTEXT = 64
print(f"{len(train_text):,} training characters | {len(CHARS)} different ones: {''.join(CHARS)!r}")


def lm_batch(ids, n=64):
    """n random windows of CONTEXT characters, and their targets."""
    starts = torch.randint(0, len(ids) - CONTEXT - 1, (n,))
    x = torch.stack([ids[s:s + CONTEXT] for s in starts])
    # TODO: y is the same windows moved one character to the right: ids[s + 1 : s + CONTEXT + 1].
    # مهمة: `y` هي النوافذ نفسها مُزاحةً حرفًا واحدًا إلى اليمين: `ids[s + 1 : s + CONTEXT + 1]`.
    return x, y


def as_string(ids):
    return "".join(CHARS[i] for i in ids)


x, y = lm_batch(train_ids, n=1)
print(f"x: {as_string(x[0][:30])!r}\ny: {as_string(y[0][:30])!r}")
TARGETS_SHIFTED = torch.equal(x[0][1:], y[0][:-1])

### Task 3.3 — a tiny GPT

The model is the review encoder with three changes:

- it embeds **characters** (44 of them) instead of words;
- it runs the layers **with the causal mask** — pass `mask=` and `is_causal=True`;
- it keeps **every** position's output and turns each into 44 scores — a guess for the next character.
  There is no averaging: every position is a prediction.

(PyTorch also has an `nn.TransformerDecoder`, but that is the decoder of the original translation model,
which also reads an encoder. A GPT-style decoder is exactly what you are building: encoder layers plus
the causal mask.)

The loss is the cross-entropy of all 64 guesses in every window. Before training, a model that knows
nothing is guessing among 44 characters, a loss of `ln 44 ≈ 3.78`. Train for 1,000 steps (about a
minute) and watch the loss on the **test** reviews fall well below that.

<div dir="rtl" align="right">

### المهمة ٣٫٣ — GPT صغير

النموذج هو مُرمِّز المراجعات بثلاثة تغييرات:

- يُضمِّن **الحروف** (٤٤ حرفًا) بدل الكلمات؛
- يُشغّل الطبقات **بالقناع السببي** — مرّر `mask=` و`is_causal=True`؛
- يحتفظ بمخرَج **كل** موضع ويحوّل كلًّا منها إلى ٤٤ درجة — تخمينًا للحرف التالي. لا متوسط هنا: كل موضع تنبّؤ.

(في PyTorch أيضًا `nn.TransformerDecoder`، لكنه مُفكِّك نموذج الترجمة الأصلي، الذي يقرأ مُرمِّزًا أيضًا. أمّا المُفكِّك على
طريقة GPT فهو بالضبط ما تبنيه: طبقات مُرمِّز مع القناع السببي.)

والخسارة هي الإنتروبيا التقاطعية لتخمينات النافذة الأربعة والستين كلها. وقبل التدريب، النموذج الذي لا يعرف شيئًا
يُخمّن بين ٤٤ حرفًا، أي خسارة `ln 44 ≈ 3.78`. درّبه ١٬٠٠٠ خطوة (نحو دقيقة) وراقب الخسارة على مراجعات **الاختبار**
تهبط تحت ذلك بكثير.

</div>

In [ ]:
class TinyGPT(nn.Module):
    def __init__(self, d_model=64, n_heads=4, n_layers=2):
        super().__init__()
        self.chars = nn.Embedding(len(CHARS), d_model)
        self.positions = nn.Embedding(CONTEXT, d_model)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, 4 * d_model, dropout=0.0,
                                           activation="gelu", batch_first=True)
        self.layers = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.out = nn.Linear(d_model, len(CHARS))

    def forward(self, x):                                     # x: (batch, length) character ids
        length = x.shape[1]
        mask = nn.Transformer.generate_square_subsequent_mask(length)
        # TODO: Add character and position embeddings, run self.layers with mask=mask and is_causal=True, and return self.out of the result: (batch, length, 44) scores for the next character.
        # مهمة: اجمع تضمين الحروف وتضمين المواضع، وشغّل `self.layers` مع `mask=mask` و`is_causal=True`، وأعِد `self.out` للنتيجة: `(batch, length, 44)` درجة للحرف التالي.


def lm_loss(model, x, y):
    scores = model(x)                                        # (batch, CONTEXT, 44)
    return nn.functional.cross_entropy(scores.reshape(-1, len(CHARS)), y.reshape(-1))


UNIFORM_LOSS = math.log(len(CHARS))
torch.manual_seed(SEED)
gpt = TinyGPT()
optimiser = torch.optim.Adam(gpt.parameters(), lr=3e-3)
LM_LOSSES = []
start = time.perf_counter()
for step in range(1_000):
    loss = lm_loss(gpt, *lm_batch(train_ids))
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
    if step % 200 == 0 or step == 999:
        with torch.no_grad():
            test_loss = float(lm_loss(gpt, *lm_batch(test_ids, n=256)))
        LM_LOSSES.append((step, round(loss.item(), 3), round(test_loss, 3)))
        print(f"  step {step:>4}: train loss {loss.item():.3f} | test loss {test_loss:.3f}")
LM_FIT_S = time.perf_counter() - start
LM_TEST_LOSS = LM_LOSSES[-1][2]
print(f"\nguessing blindly among {len(CHARS)} characters: {UNIFORM_LOSS:.2f} | the model: {LM_TEST_LOSS:.2f}  ({LM_FIT_S:.0f} s)")

### Task 3.4 — let it write

Generation is a loop. Give the model a prompt, take its scores for the **last** position, turn them into
probabilities, draw one character, append it, and go again — exactly how tomorrow's LLM answers you.

The draw uses a **temperature**: the scores are divided by it before the softmax. Below 1 the likely
characters get likelier and the text is safer; above 1 the distribution flattens and the text gets
stranger. Write the line that draws the next character, then try a few temperatures.

It will not make sense — a minute of training on a laptop, one character at a time. But look at what it
learned from nothing: spaces between word-shaped pieces, real words, punctuation, the way reviews start.

<div dir="rtl" align="right">

### المهمة ٣٫٤ — دعه يكتب

التوليد حلقة. أعطِ النموذج بداية، وخُذ درجاته للموضع **الأخير**، وحوّلها إلى احتمالات، واسحب حرفًا واحدًا، وألحِقه،
وكرّر — تمامًا كما يُجيبك نموذج اللغة الكبير غدًا.

ويستعمل السحب **حرارة** (temperature): تُقسَم الدرجات عليها قبل softmax. تحت ١ يصير المحتمَل أكثر احتمالًا ويصير
النص أكثر أمانًا؛ وفوق ١ يتسطّح التوزيع ويصير النص أغرب. اكتب السطر الذي يسحب الحرف التالي، ثم جرّب بضع حرارات.

لن يكون للنص معنى — دقيقة من التدريب على حاسوب محمول، حرفًا بعد حرف. لكن انظر إلى ما تعلّمه من لا شيء: مسافات بين
قطع على هيئة كلمات، وكلمات حقيقية، وعلامات ترقيم، والطريقة التي تبدأ بها المراجعات.

</div>

In [ ]:
def generate(model, prompt, n_chars=150, temperature=0.8):
    model.eval()
    ids = torch.tensor([CHAR_ID[c] for c in prompt])
    with torch.no_grad():
        for _ in range(n_chars):
            scores = model(ids[-CONTEXT:][None])[0, -1]        # the guess after the last character
            probabilities = torch.softmax(scores / temperature, dim=0)
            # TODO: Draw one character id from `probabilities` with torch.multinomial(probabilities, 1).
            # مهمة: اسحب رقم حرف واحد من `probabilities` بـ`torch.multinomial(probabilities, 1)`.
            ids = torch.cat([ids, next_id])
    return as_string(ids)


torch.manual_seed(0)
SAMPLES = {}
for temperature in [0.5, 0.8, 1.5]:
    SAMPLES[temperature] = generate(gpt, "the food was", temperature=temperature)
    print(f"temperature {temperature}:\n  {SAMPLES[temperature]!r}\n")

## Section 4 — Stretch  (≈25 min)

**A. Take the mask away.** Train the same tiny GPT for 200 steps **without** the causal mask (pass no
`mask` and `is_causal=False` — subclass it or copy the class). The training loss collapses far below
anything the real model reached: every position can now simply read the next character. Then ask it to
write, and watch it fall apart — at generation time there is no next character to read. A beautiful
loss curve and a useless model: that is what the mask prevents.

**B. The memory task, last round.** Tuesday's task at length 100, where the recurrent cells were a
lottery. The cells below train a two-layer encoder on it and compare it with Tuesday's table. The last
position can reach position 0 directly, in one step — nothing has to survive a chain or fit in a window.

<div dir="rtl" align="right">

## القسم الرابع — إضافي (نحو ٢٥ دقيقة)

**أ. انزع القناع.** درّب GPT الصغير نفسه ٢٠٠ خطوة **بدون** القناع السببي (لا تمرّر `mask` واجعل `is_causal=False` —
بصنف فرعي أو بنسخ الصنف). فتهوي خسارة التدريب إلى ما دون أي شيء بلغه النموذج الحقيقي بكثير: فكل موضع يستطيع الآن أن
يقرأ الحرف التالي ببساطة. ثم اطلب منه أن يكتب، وراقبه ينهار — فعند التوليد لا يوجد حرف تالٍ ليقرأه. منحنى خسارة
جميل ونموذج عديم الفائدة: هذا ما يمنعه القناع.

**ب. مهمة الذاكرة، الجولة الأخيرة.** مهمة الثلاثاء بالطول ١٠٠، حيث كانت الخلايا المتكرّرة يانصيبًا. تُدرّب الخلايا
أدناه مُرمِّزًا من طبقتين عليها وتقارنه بجدول الثلاثاء. فالموضع الأخير يستطيع بلوغ الموضع ٠ مباشرة، في خطوة واحدة —
لا شيء يجب أن ينجو من سلسلة أو يتّسع في نافذة.

</div>

In [ ]:
# The memory task: the label is the first token; every token after it is random noise.
N_CLASSES, MEMORY_VOCAB = 4, 12          # tokens 0-3 can be labels, 4-11 are distractors


def memory_batch(n, length, generator):
    x = torch.randint(N_CLASSES, MEMORY_VOCAB, (n, length), generator=generator)
    y = torch.randint(0, N_CLASSES, (n,), generator=generator)
    x[:, 0] = y
    return x, y


def train_memory(model, length, steps=800, lr=0.01, seed=0):
    """Train on fresh random batches, then score 1,000 held-out sequences (chance is 0.25)."""
    torch.manual_seed(seed)
    generator = torch.Generator().manual_seed(seed)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(steps):
        x, y = memory_batch(64, length, generator)
        loss = nn.functional.cross_entropy(model(x), y)
        optimiser.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimiser.step()
    x, y = memory_batch(1000, length, torch.Generator().manual_seed(999))
    with torch.no_grad():
        return float((model(x).argmax(1) == y).float().mean())

In [ ]:
class TransformerMemory(nn.Module):
    def __init__(self, d_model=32, n_heads=2, n_layers=2, length=100):
        super().__init__()
        self.tokens = nn.Embedding(MEMORY_VOCAB, d_model)
        self.positions = nn.Embedding(length, d_model)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, 4 * d_model, dropout=0.0,
                                           activation="gelu", batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers, enable_nested_tensor=False)
        self.out = nn.Linear(d_model, N_CLASSES)

    def forward(self, x):
        h = self.tokens(x) + self.positions(torch.arange(x.shape[1]))
        return self.out(self.encoder(h)[:, -1])               # answer from the last position


MEMORY_RESULTS = dict(json.loads(load_artefact("memory_task_D2.json").read_text(encoding="utf-8")))
start = time.perf_counter()
torch.manual_seed(0)
TRANSFORMER_MEMORY = train_memory(TransformerMemory(), 100, lr=0.003)
MEMORY_RESULTS["transformer @ 100 (seed 0)"] = TRANSFORMER_MEMORY
print(f"transformer, length 100: accuracy {TRANSFORMER_MEMORY:.3f}  ({time.perf_counter() - start:.0f} s)\n")
print(pd.Series(MEMORY_RESULTS).round(3).to_string())

## Save your artefacts

- `leaderboard_D4.json` — the encoder's row.
- `language_model_D4.json` — the language model's losses and what it wrote.
- `memory_task_D4.json` — the complete memory table, if you ran stretch B.

<div dir="rtl" align="right">

## احفظ آثارك

- `leaderboard_D4.json` — صفّ المُرمِّز.
- `language_model_D4.json` — خسائر نموذج اللغة وما كتبه.
- `memory_task_D4.json` — جدول الذاكرة الكامل، إن شغّلت الإضافي (ب).

</div>

In [ ]:
(ARTEFACT_DIR / "leaderboard_D4.json").write_text(json.dumps(BOARD_D4, indent=2), encoding="utf-8")
(ARTEFACT_DIR / "language_model_D4.json").write_text(json.dumps({
    "characters": len(CHARS), "context": CONTEXT, "uniform_loss": round(UNIFORM_LOSS, 3),
    "losses": LM_LOSSES, "test_loss": LM_TEST_LOSS, "fit_s": round(LM_FIT_S, 1),
    "samples": {str(t): s for t, s in SAMPLES.items()},
}, indent=2), encoding="utf-8")
if "TRANSFORMER_MEMORY" in globals():
    (ARTEFACT_DIR / "memory_task_D4.json").write_text(json.dumps(MEMORY_RESULTS, indent=2), encoding="utf-8")
print("wrote leaderboard_D4.json, language_model_D4.json" + (" and memory_task_D4.json" if "TRANSFORMER_MEMORY" in globals() else ""))

## Sanity check

<div dir="rtl" align="right">

## فحص سلامة

</div>

In [ ]:
check(np.allclose(WEIGHTS, [0.42, 0.16, 0.42], atol=0.005) and np.allclose(OUTPUT, [0.84, 0.58], atol=0.005)
      and np.allclose(FLIPPED_WEIGHTS, [0.16, 0.42, 0.42], atol=0.005) and ORDER_BLIND,
      "attention must reproduce the slide (0.42/0.16/0.42, [0.84, 0.58], flipped 0.16/0.42/0.42) and ignore the order",
      "يجب أن يُعيد الانتباه إنتاج الشريحة (٠٫٤٢/٠٫١٦/٠٫٤٢، `[0.84, 0.58]`، والمقلوبة ٠٫١٦/٠٫٤٢/٠٫٤٢) وألّا يبالي بالترتيب")
check(SLIDE_MATCH,
      "nn.MultiheadAttention with its projections switched off must give the warm-up softened by sqrt(2)",
      "يجب أن تُعطي `nn.MultiheadAttention` بإسقاطات مُطفأة الإحماءَ مليّنًا بـ`sqrt(2)`")
check(tuple(tokens_out.shape) == tuple(tokens_in.shape),
      f"the encoder must give back the shape it was given — got {tuple(tokens_out.shape)}",
      f"يجب أن يُعيد المُرمِّز الشكل الذي أُعطيه — والناتج {tuple(tokens_out.shape)}")
check(ENCODER_ACCURACY > 0.75,
      f"the review encoder must score above 0.75 on the test set — got {ENCODER_ACCURACY:.4f}",
      f"يجب أن يتجاوز مُرمِّز المراجعات ٠٫٧٥ على الاختبار — والناتج {ENCODER_ACCURACY:.4f}")
check(IS_LOWER_TRIANGULAR and ROWS_SUM_TO_ONE,
      "the causal weights must be zero above the diagonal, with every row still summing to 1",
      "يجب أن تكون الأوزان السببية صفرًا فوق القُطر، وأن يبقى كل صف يجمع إلى ١")
check(TARGETS_SHIFTED,
      "each target window must be its input window moved one character to the right",
      "يجب أن تكون كل نافذة هدف هي نافذة مُدخلها مُزاحةً حرفًا واحدًا إلى اليمين")
check(LM_TEST_LOSS < 0.6 * UNIFORM_LOSS,
      f"the language model's test loss must fall well below blind guessing ({UNIFORM_LOSS:.2f}) — got {LM_TEST_LOSS:.2f}",
      f"يجب أن تهبط خسارة نموذج اللغة على الاختبار تحت التخمين الأعمى ({UNIFORM_LOSS:.2f}) بكثير — والناتج {LM_TEST_LOSS:.2f}")
report()

## What's next

**W5D5 — LLMs.** Today the same block did two jobs. As an **encoder** it read reviews — and, trained from
scratch, it did not beat Tuesday's GRU: architecture alone is not enough. As a **decoder** with the
causal mask it learned to write, one character at a time, from a minute of training.

Tomorrow you run a decoder built exactly like today's — with subword tokens instead of characters, 24
layers instead of 2, and **pretrained** on trillions of tokens. You will look inside its next-token
probabilities, write the sampling loop you wrote today, and use it to classify the week's reviews
**without training it at all**. Then the leaderboard is complete, and you decide which model you would
ship.

<div dir="rtl" align="right">

## ما التالي

**الأسبوع ٥ اليوم ٥ — نماذج اللغة الكبيرة.** قامت الكتلة نفسها اليوم بعملين. **مُرمِّزًا** قرأت المراجعات — ولم
تتفوّق، مُدرَّبةً من الصفر، على GRU الثلاثاء: فالمعمارية وحدها لا تكفي. و**مُفكِّكًا** بالقناع السببي تعلّمت أن تكتب،
حرفًا بعد حرف، من دقيقة تدريب.

وغدًا تُشغّل مُفكِّكًا مبنيًا تمامًا كمُفكِّك اليوم — برموز جزئية بدل الحروف، و٢٤ طبقة بدل طبقتين، و**مُدرَّبًا مسبقًا**
على تريليونات الرموز. ستنظر داخل احتمالاته للرمز التالي، وتكتب حلقة أخذ العيّنات التي كتبتها اليوم، وتستعمله لتصنيف
مراجعات الأسبوع **دون أي تدريب**. ثم تكتمل اللوحة، وتقرّر أيّ نموذج كنت لتنشره.

</div>